In [1]:
# ============================================================
# NOTEBOOK 5 — BEHAVIOUR SEGMENT
# Assign 9-name behaviour segment from percentile ranks.
# Rules are mutually exclusive and exhaustive by construction.
# ============================================================

In [2]:
import pandas as pd
import numpy as np
import os

In [3]:
# Assigning path
path = r'/Users/elia/Desktop/[02] Data Projects/[2] Working Folder/[3] InstaCart Store'

In [4]:
# ------------------------------------------------------------
# LOAD CHECKPOINT FROM STEP 4
# ------------------------------------------------------------
df_customers_agg = pd.read_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step4.pkl')
)

In [5]:
# Verify the columns we need are all present

In [6]:
required_cols = [
    'user_id', 'unique_orders', 'unique_products', 'avg_product_price',
    'reorder_rate', 'orders_pct_rank', 'reorder_pct_rank',
    'products_pct_rank', 'price_pct_rank', 'behaviour_score', 'priority_level'
]
for col in required_cols:
    assert col in df_customers_agg.columns, f"Missing required column: {col}"

print(f"✅ Checkpoint loaded: {df_customers_agg.shape}")
print()

✅ Checkpoint loaded: (206209, 13)



In [7]:
# ============================================================
# 5A — BEHAVIOUR SEGMENT FUNCTION
# ============================================================
# Rules fire top-to-bottom. First match wins. No silent overrides.
# Final 'Standard Customers' is a NAMED bucket for middling profiles —
# not a catch-all rename of unhandled cases.

In [8]:
def assign_behaviour_segment(row):
    """
    Assign a behaviour segment from percentile ranks.
    
    Inputs (all in [0, 1], from row):
        orders_pct_rank    — frequency rank
        reorder_pct_rank   — loyalty rank
        products_pct_rank  — basket diversity rank
        price_pct_rank     — premium-vs-value rank
    
    Returns one of 9 segment labels. Function is exhaustive:
    every input combination produces a label; the final return
    is a legitimate "Standard Customers" bucket, not a fallback patch.
    """
    o = row['orders_pct_rank']
    r = row['reorder_pct_rank']
    p = row['products_pct_rank']
    pr = row['price_pct_rank']
    
    # Rule 1: Champions — top tier on frequency AND loyalty
    if o >= 0.80 and r >= 0.80:
        return 'Champions'
    
    # Rule 2: Loyal Repeat Buyers — high frequency and strong loyalty
    if o >= 0.70 and r >= 0.60:
        return 'Loyal Repeat Buyers'
    
    # Rule 3: Frequent Explorers — high frequency, broad basket, LOW loyalty
    if o >= 0.70 and p >= 0.70 and r < 0.50:
        return 'Frequent Explorers'
    
    # Rule 4: Premium Basket Buyers — high price AND broad basket
    if pr >= 0.80 and p >= 0.60:
        return 'Premium Basket Buyers'
    
    # Rule 5: Upsell Candidates — moderate frequency + premium lean
    if o >= 0.60 and pr >= 0.60:
        return 'Upsell Candidates'
    
    # Rule 6: Developing Repeat Buyers — moderate frequency + decent loyalty
    if o >= 0.40 and r >= 0.50:
        return 'Developing Repeat Buyers'
    
    # Rule 7: Low Engagement — bottom tier on both frequency and loyalty
    if o < 0.30 and r < 0.30:
        return 'Low Engagement Customers'
    
    # Rule 8: Light Buyers — low frequency but not bottom-bottom
    if o < 0.40:
        return 'Light Buyers'
    
    # Rule 9: Standard Customers — moderate everything, no dominant trait
    return 'Standard Customers'

In [9]:
# ============================================================
# 5B — APPLY FUNCTION
# ============================================================
# .apply with axis=1 walks each row through the function.
# This is slower than vectorised np.select but more readable
# and only runs once on a 206k-row table — completes in seconds.

In [10]:
df_customers_agg['behaviour_segment'] = df_customers_agg.apply(
    assign_behaviour_segment, axis=1
)

# Set as ordered categorical for consistent sorting in Power BI
behaviour_segment_order = [
    'Champions',
    'Loyal Repeat Buyers',
    'Frequent Explorers',
    'Premium Basket Buyers',
    'Upsell Candidates',
    'Developing Repeat Buyers',
    'Standard Customers',
    'Light Buyers',
    'Low Engagement Customers'
]

df_customers_agg['behaviour_segment'] = pd.Categorical(
    df_customers_agg['behaviour_segment'],
    categories=behaviour_segment_order,
    ordered=True
)

In [11]:
# ============================================================
# 5C — VALIDATION
# ============================================================

In [12]:
# No nulls
assert df_customers_agg['behaviour_segment'].isnull().sum() == 0, \
    "Some customers have no behaviour_segment — function is not exhaustive"

# All values must be in the expected list
expected = set(behaviour_segment_order)
actual = set(df_customers_agg['behaviour_segment'].astype(str).unique())
assert actual.issubset(expected), \
    f"Unexpected segment values: {actual - expected}"

# Cross-check Rule 1: every Champion has both ranks >= 0.80
champions = df_customers_agg[df_customers_agg['behaviour_segment'] == 'Champions']
assert (champions['orders_pct_rank'] >= 0.80).all()
assert (champions['reorder_pct_rank'] >= 0.80).all()

# Cross-check Rule 7: every Low Engagement has both ranks < 0.30
low_eng = df_customers_agg[df_customers_agg['behaviour_segment'] == 'Low Engagement Customers']
assert (low_eng['orders_pct_rank'] < 0.30).all()
assert (low_eng['reorder_pct_rank'] < 0.30).all()

# Total must equal customer count
assert df_customers_agg.shape[0] == 206209

print("✅ All Step 5 assertions passed")
print()

✅ All Step 5 assertions passed



In [13]:
# ============================================================
# 5D — DESCRIPTIVE OUTPUT
# ============================================================

In [14]:
print("Behaviour segment distribution:")
print(df_customers_agg['behaviour_segment'].value_counts().sort_index().to_string())
print()

print("Behaviour segment distribution (%):")
print((df_customers_agg['behaviour_segment'].value_counts(normalize=True) * 100)
      .sort_index().round(2).to_string())
print()

print("Behaviour segment × priority level cross-tab:")
print(pd.crosstab(
    df_customers_agg['behaviour_segment'],
    df_customers_agg['priority_level']
))
print()

Behaviour segment distribution:
behaviour_segment
Champions                   25628
Loyal Repeat Buyers         24695
Frequent Explorers           4368
Premium Basket Buyers        5900
Upsell Candidates            7842
Developing Repeat Buyers    30020
Standard Customers          23501
Light Buyers                44458
Low Engagement Customers    39797

Behaviour segment distribution (%):
behaviour_segment
Champions                   12.43
Loyal Repeat Buyers         11.98
Frequent Explorers           2.12
Premium Basket Buyers        2.86
Upsell Candidates            3.80
Developing Repeat Buyers    14.56
Standard Customers          11.40
Light Buyers                21.56
Low Engagement Customers    19.30

Behaviour segment × priority level cross-tab:
priority_level            Low Priority  Maintain  Growth Priority  \
behaviour_segment                                                   
Champions                            0         0             6456   
Loyal Repeat Buyers          

In [17]:
# Save df_customers_agg as a checkpoint after Step 5

df_customers_agg.to_pickle(
    os.path.join(path, '[2] Data', '[2] Prepared Data', 'customers_agg_step5.pkl')
)
print(f"✅ Checkpoint saved: customers_agg_step5.pkl")
print(f"   Shape: {df_customers_agg.shape}")
print(f"   Columns: {df_customers_agg.columns.tolist()}")

✅ Checkpoint saved: customers_agg_step5.pkl
   Shape: (206209, 14)
   Columns: ['user_id', 'unique_orders', 'unique_products', 'avg_product_price', 'total_reordered_items', 'total_line_items', 'reorder_rate', 'orders_pct_rank', 'reorder_pct_rank', 'products_pct_rank', 'price_pct_rank', 'behaviour_score', 'priority_level', 'behaviour_segment']
